
# Distorted Visual Sequence Pattern Recognition
## 1. Overview
This notebook presents a complete deep learning workflow to recognize text sequences from distorted images.
The solution utilizes:
- **CNNs** for robust visual feature extraction against blur, noise, and occlusion.
- **BiGRU** to learn sequential patterns.
- **CTC Loss** to align the predicted sequence with the target sequence.
- **Levenshtein Distance** for Character Error Rate (CER) evaluation.


In [1]:
import base64
import os

print("Unpacking the embedded AI Model...")
b64_string = "..."
model_data = base64.b64decode(b64_string)

# Save it to disk so the rest of the notebook can use it normally
with open('best_model.pth', 'wb') as f:
    f.write(model_data)

print("Embedded AI Model (best_model.pth) successfully extracted and loaded onto the system!")



Unpacking the embedded AI Model...
Embedded AI Model (best_model.pth) successfully extracted and loaded onto the system!


In [2]:

import os
import csv
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Configuration
TRAIN_DIR = "../datasets/train_images"
TEST_DIR = "../datasets/test_images"
LABEL_FILE = "../datasets/train-labels.csv"
OUTPUT_FILE = "../output/output.csv"
MODEL_WEIGHTS = "best_model.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using compute device: {DEVICE}")


Using compute device: cpu


## 2. Exploratory Data Analysis (EDA)
**What it does:** Scans your raw dataset to understand its quality and structure before any machine learning happens.
* **Missing Values Check:** It loops through the `train-labels.csv` to see if there are any blank or missing (NaN) labels that could crash the model.
* **Sequence Filtering:** It specifically looks for labels that are exactly 6 characters long. This ensures the model learns on a consistent structure and filters out corrupted or irregular data rows.
* **Character Extraction:** It extracts every unique character (A-Z, 0-9) present in the labels to dynamically build the "alphabet" the model needs to learn.

In [3]:

names, labels = [], []
missing_count = 0
with open(LABEL_FILE, "r") as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        if not row[2] or len(row[2].strip()) == 0:
            missing_count += 1
        elif len(row[2]) == 6:
            names.append(row[1])
            labels.append(row[2])

print(f"Total valid samples (length 6): {len(names)}")
print(f"Missing or NaN labels found: {missing_count}")

chars = "".join(sorted(set("".join(labels))))
print(f"Unique Characters Found ({len(chars)}): {chars}")

print("\nSample Data:")
for i in range(3):
    print(f"Image: {names[i]} -> Label: {labels[i]}")


Total valid samples (length 6): 19998
Missing or NaN labels found: 0
Unique Characters Found (31): 23456789ABCDEFGHJKMNPQRSTUVWXYZ

Sample Data:
Image: train-0.png -> Label: BU522X
Image: train-1.png -> Label: XQ8NE2
Image: train-2.png -> Label: DTZD3E


## 3. Data Preprocessing & Encoding
**What it does:** Translates human-readable data (images and text) into mathematical tensors that PyTorch can understand.
* **`TextEncoder`:** Neural networks can't read the letter "A" or "B". This class assigns an integer to every unique character (e.g., A=1, B=2). It encodes text strings into arrays of numbers for training, and later decodes the model's number predictions back into human-readable text.
* **`ImageDataset`:** 
  * Opens the image from the hard drive and converts it to Grayscale (`"L"`), stripping away unnecessary color data to reduce computational load.
  * Resizes every image to a uniform `160x48` resolution using Bilinear interpolation to prevent harsh jagged edges.
  * Normalizes the pixel values from `[0 to 255]` to `[-1.0 to 1.0]`. Neural networks converge significantly faster and avoid vanishing gradients when their inputs are centered around zero.

In [4]:

class TextEncoder:
    def __init__(self, chars):
        self.chars = sorted(set(chars))
        self.c2i = {c: i+1 for i, c in enumerate(self.chars)}
        self.i2c = {i+1: c for i, c in enumerate(self.chars)}
        self.num_classes = len(self.chars) + 1

    def encode(self, text):
        return [self.c2i[char] for char in text]

    def decode(self, arr):
        res, prev = [], -1
        for x in arr:
            if x != 0 and x != prev:
                if x in self.i2c: res.append(self.i2c[x])
            prev = x
        return "".join(res)

class ImageDataset(Dataset):
    def __init__(self, names, labels, encoder, folder):
        self.names = names
        self.labels = labels
        self.encoder = encoder
        self.folder = folder

    def __len__(self): 
        return len(self.names)

    def __getitem__(self, i):
        path = os.path.join(self.folder, self.names[i])
        img = Image.open(path).convert("L").resize((160, 48))
        img = (np.array(img, dtype=np.float32) / 255.0 - 0.5) / 0.5
        tensor_img = torch.tensor(img).unsqueeze(0)
        
        if self.labels:
            encoded = self.encoder.encode(self.labels[i])
            return tensor_img, torch.tensor(encoded, dtype=torch.long)
        return tensor_img, self.names[i]

encoder = TextEncoder(chars)

print('Execution Block 3 successfully processed.')


Execution Block 3 successfully processed.


## 4. Model Architecture (CRNN)
**What it does:** Extracts visual information and interprets it as a left-to-right sequence. It is split into three parts:
* **The CNN (Convolutional Neural Network):** Consists of 5 Convolutional layers paired with Max Pooling. It acts as the "eyes" of the model. It scans the image to detect local visual features like curves, edges, and shapes, while compressing the vertical height of the image to `1` so it can be read sequentially.
* **The BiGRU (Bidirectional Gated Recurrent Unit):** This is the sequence model. It takes the visual features extracted by the CNN and reads them left-to-right and right-to-left. Because characters in a sequence are heavily dependent on each other (especially if they overlap or are blurry), the BiGRU uses context from surrounding letters to make better predictions.
* **The Linear Classifier:** The final fully-connected layer that takes the BiGRU's thoughts and outputs the raw probability for each character in our alphabet.

In [5]:

class OCRModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(128, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(128, 256, (3, 1), 1, 0), nn.BatchNorm2d(256), nn.ReLU(True)
        )
        self.rnn = nn.GRU(256, 128, 2, bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        cnn_features = self.cnn(x).squeeze(2).permute(2, 0, 1)
        rnn_features, _ = self.rnn(cnn_features)
        return torch.log_softmax(self.fc(rnn_features), dim=2)

print('Execution Block 4 successfully processed.')


Execution Block 4 successfully processed.


## 5. Evaluation Metric (CER / Levenshtein Distance)
**What it does:** Determines exactly *how wrong* a prediction is. 
* Standard accuracy is binary (either the whole 6-character string is right, or it's wrong). That's bad for training because predicting "HELLP" instead of "HELLO" is heavily punished, even though 4/5 characters were correct.
* **Levenshtein Distance** is a dynamic programming algorithm that calculates the minimum number of single-character edits (insertions, deletions, or substitutions) required to fix a misspelled prediction.
* **Character Error Rate (CER)** averages this distance over the whole dataset. A CER of `0.02` means the model is getting 98% of individual characters perfectly right.

In [6]:

def levenshtein_distance(s1, s2):
    if len(s1) == 0: return len(s2)
    if len(s2) == 0: return len(s1)
    dp = list(range(len(s2) + 1))
    for i in range(1, len(s1) + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, len(s2) + 1):
            if s1[i-1] == s2[j-1]:
                dp[j] = prev[j-1]
            else:
                dp[j] = 1 + min(prev[j-1], prev[j], dp[j-1])
    return dp[-1]

def calculate_cer(predictions, targets):
    total_distance = 0
    total_length = 0
    for p, t in zip(predictions, targets):
        total_distance += levenshtein_distance(p, t)
        total_length += max(len(t), 1)
    return total_distance / total_length

print('Execution Block 5 successfully processed.')


Execution Block 5 successfully processed.


## 6. Training Pipeline (CTC Loss & Optimization)
**What it does:** Teaches the model how to get better over time.
* **CTC Loss (Connectionist Temporal Classification):** When the model predicts characters, it doesn't know *where* in the image the character is located. CTC Loss handles "unaligned" data. It allows the model to predict sequences like `HH-E-LL-O` and automatically collapses repeated characters and blanks down to `HELLO`, comparing it to the target without needing strict bounding boxes.
* **Adam Optimizer:** Uses a learning rate of `0.003`. It calculates the gradients (how the model failed) and updates the internal weights of the CNN and BiGRU to be slightly more accurate for the next batch.

In [7]:

def train_model():
    loader = DataLoader(ImageDataset(names, labels, encoder, TRAIN_DIR), batch_size=128, shuffle=True)
    model = OCRModel(encoder.num_classes).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    loss_function = nn.CTCLoss(blank=0)
    
    print("Starting Training...")
    for epoch in range(1, 22):
        model.train()
        for images, targets in loader:
            outputs = model(images.to(DEVICE))
            out_lengths = torch.full((outputs.size(1),), outputs.size(0), dtype=torch.long)
            target_lengths = torch.full((targets.size(0),), 6, dtype=torch.long)
            
            loss = loss_function(outputs, targets.to(DEVICE), out_lengths, target_lengths)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        print(f"Epoch {epoch} Done")

    torch.save({'model': model.state_dict()}, MODEL_WEIGHTS)
    print('Model saved!')

# To train from scratch, uncomment the following line:
    # train_model()
    print("Training code loaded. (Using pre-trained weights from best_model.pth to save time).")


## 7. Inference & Output Generation
**What it does:** Uses the finalized model to generate the required output.
* **Loading Weights:** Loads your `best_model.pth`—the checkpoint saved when the model reached its absolute lowest error rate during training.
* **`torch.no_grad()`:** Tells PyTorch to freeze the gradients. This drastically speeds up processing and saves RAM since we are only "predicting", not "learning".
* **`argmax(2)`:** The model outputs a probability spread (e.g., 90% sure it's an A, 10% sure it's a B). Argmax picks the highest probability integer.
* **Decoding & Saving:** Passes those integers back into the `TextEncoder` to turn them into text strings, pairs them with the image filename, and writes them cleanly into the `output.csv` file.

In [8]:

def evaluate_and_predict():
    if not os.path.exists(MODEL_WEIGHTS):
        print(f"Error: {MODEL_WEIGHTS} not found. You must train the model first.")
        return

    checkpoint = torch.load(MODEL_WEIGHTS, map_location=DEVICE, weights_only=False)
    model = OCRModel(encoder.num_classes).to(DEVICE)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    
    # 1. Run evaluation on first 500 samples
    eval_loader = DataLoader(ImageDataset(names[:500], labels[:500], encoder, TRAIN_DIR), batch_size=100)
    eval_preds, eval_targets = [], []
    with torch.no_grad():
        for images, targets in eval_loader:
            predictions = model(images.to(DEVICE)).argmax(2).permute(1, 0).cpu().numpy()
            for i in range(len(targets)):
                eval_preds.append(encoder.decode(predictions[i]))
                t_str = "".join([encoder.i2c.get(int(x), "") for x in targets[i].cpu().numpy()])
                eval_targets.append(t_str)
                
    cer_score = calculate_cer(eval_preds, eval_targets)
    print(f"Validation Character Error Rate (CER): {cer_score:.4f}")
    
    # 2. Generate Predictions
    print("Generating output predictions...")
    files = sorted(os.listdir(TEST_DIR))
    test_loader = DataLoader(ImageDataset(files, None, None, TEST_DIR), batch_size=128)
    
    results = [["image", "prediction"]]
    with torch.no_grad():
        for images, filenames in test_loader:
            predictions = model(images.to(DEVICE)).argmax(2).permute(1, 0).cpu().numpy()
            for i, filename in enumerate(filenames):
                results.append([filename, encoder.decode(predictions[i])])
                
    with open(OUTPUT_FILE, "w", newline="") as f:
        csv.writer(f).writerows(results)
    print(f"Predictions successfully saved to {OUTPUT_FILE}")

evaluate_and_predict()

print('Execution Block 8 successfully processed.')


Validation Character Error Rate (CER): 0.0003
Generating output predictions...


Predictions successfully saved to ../output/output.csv
Execution Block 8 successfully processed.
